# 🎬 Text-to-Video Generator - Fixed Version

Generate videos from text prompts using AI models

## 🚀 Quick Start
1. Run all cells below (Runtime → Run all)
2. Enter your text prompt in the UI
3. Click "Generate Video" button
4. Download your video!

## ⚙️ Requirements
- GPU Runtime (Runtime → Change runtime type → GPU)
- ~8GB VRAM minimum

## 📦 Step 1: Install Dependencies

In [ ]:
# Install required packages
!pip install -q --upgrade torch torchvision torchaudio
!pip install -q --upgrade transformers diffusers accelerate gradio
!pip install -q --upgrade xformers
!pip install -q opencv-python imageio-ffmpeg

print("✅ All packages installed successfully!")

## 🔧 Step 2: Import Libraries

In [ ]:
import torch
import numpy as np
import gradio as gr
import os
from datetime import datetime
from PIL import Image
import warnings
warnings.filterwarnings('ignore')

# Check GPU availability
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"🎯 Using device: {device}")

if device == "cuda":
    print(f"📊 GPU: {torch.cuda.get_device_name(0)}")
    vram_gb = torch.cuda.get_device_properties(0).total_memory / 1024**3
    print(f"💾 VRAM: {vram_gb:.1f} GB")
    
    if vram_gb < 8:
        print("⚠️  Warning: VRAM is less than 8GB. Generation may be slow or fail.")
else:
    print("⚠️  Warning: No GPU detected. Please enable GPU in Runtime settings!")

## 🎥 Step 3: Load Text-to-Video Model

In [ ]:
print("📥 Loading text-to-video model...")
print("⏳ This may take 3-5 minutes on first run...")

try:
    from diffusers import DiffusionPipeline, DPMSolverMultistepScheduler
    
    # Use ModelScope text-to-video model (reliable and tested)
    model_id = "damo-vilab/text-to-video-ms-1.7b"
    
    pipe = DiffusionPipeline.from_pretrained(
        model_id,
        torch_dtype=torch.float16,
        variant="fp16"
    )
    
    # Move to GPU
    pipe = pipe.to(device)
    
    # Use DPM++ scheduler for faster generation
    pipe.scheduler = DPMSolverMultistepScheduler.from_config(pipe.scheduler.config)
    
    # Enable memory optimizations
    pipe.enable_model_cpu_offload()
    
    # Try to enable xformers (may fail on some systems)
    try:
        pipe.enable_xformers_memory_efficient_attention()
        print("✅ xformers memory optimization enabled")
    except:
        print("⚠️  xformers not available, using default attention")
    
    print("✅ Model loaded successfully!")
    
except Exception as e:
    print(f"❌ Error loading model: {str(e)}")
    print("\n🔧 Troubleshooting tips:")
    print("1. Make sure you have GPU enabled (Runtime → Change runtime type → GPU)")
    print("2. Try restarting the runtime (Runtime → Restart runtime)")
    print("3. Make sure you have enough VRAM (8GB+ recommended)")
    raise

## 🎨 Step 4: Define Video Generation Function

In [ ]:
def generate_video(
    prompt,
    negative_prompt="",
    num_frames=16,
    num_inference_steps=25,
    guidance_scale=7.5,
    seed=-1,
    fps=8
):
    """
    Generate video from text prompt
    
    Args:
        prompt: Text description of the video
        negative_prompt: Things to avoid in the video
        num_frames: Number of frames (default: 16)
        num_inference_steps: Number of denoising steps (default: 25)
        guidance_scale: How closely to follow prompt (default: 7.5)
        seed: Random seed (-1 for random)
        fps: Frames per second (default: 8)
    
    Returns:
        Path to generated video file
    """
    print(f"🎬 Generating video for: {prompt}")
    print(f"⚙️ Settings: {num_frames} frames, {num_inference_steps} steps, guidance={guidance_scale}")
    
    # Set seed for reproducibility
    if seed != -1:
        torch.manual_seed(seed)
        np.random.seed(seed)
    
    try:
        # Generate video
        with torch.no_grad():
            video_frames = pipe(
                prompt=prompt,
                negative_prompt=negative_prompt,
                num_inference_steps=num_inference_steps,
                guidance_scale=guidance_scale,
                generator=torch.Generator(device).manual_seed(seed) if seed != -1 else None
            ).frames[0]
        
        # Save video using imageio
        import imageio
        
        timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
        output_path = f"/content/video_{timestamp}.mp4"
        
        # Convert frames to video
        writer = imageio.get_writer(output_path, fps=fps)
        for frame in video_frames:
            # Convert PIL Image to numpy array if needed
            if isinstance(frame, Image.Image):
                frame = np.array(frame)
            writer.append_data(frame)
        writer.close()
        
        print(f"✅ Video saved to: {output_path}")
        print(f"📊 Video size: {os.path.getsize(output_path) / (1024*1024):.1f} MB")
        
        return output_path
        
    except Exception as e:
        print(f"❌ Error generating video: {str(e)}")
        raise

## 🖥️ Step 5: Create Simple UI

In [ ]:
# Create Gradio interface
with gr.Blocks(title="Text-to-Video Generator", theme=gr.themes.Soft()) as demo:
    gr.Markdown("""
    # 🎬 Text-to-Video Generator
    
    Generate amazing videos from text descriptions using AI!
    
    ---
    """)
    
    with gr.Row():
        with gr.Column():
            # Text prompt input
            prompt_input = gr.Textbox(
                label="✍️ Your Prompt",
                placeholder="Describe the video you want to create...",
                lines=3,
                value="A beautiful sunset over the ocean, with waves gently rolling onto a sandy beach"
            )
            
            # Negative prompt
            negative_prompt = gr.Textbox(
                label="🚫 Negative Prompt (optional)",
                placeholder="Things to avoid in the video...",
                lines=2,
                value="blurry, low quality, distorted, ugly, bad animation"
            )
            
            # Advanced settings (collapsed)
            with gr.Accordion("⚙️ Advanced Settings", open=False):
                num_frames = gr.Slider(
                    minimum=8,
                    maximum=32,
                    value=16,
                    step=1,
                    label="🎞️ Number of Frames"
                )
                
                num_steps = gr.Slider(
                    minimum=10,
                    maximum=50,
                    value=25,
                    step=1,
                    label="🔄 Inference Steps"
                )
                
                guidance_scale = gr.Slider(
                    minimum=1.0,
                    maximum=15.0,
                    value=7.5,
                    step=0.5,
                    label="🎯 Guidance Scale"
                )
                
                seed = gr.Number(
                    value=-1,
                    label="🎲 Seed (-1 for random)"
                )
                
                fps = gr.Slider(
                    minimum=4,
                    maximum=16,
                    value=8,
                    step=1,
                    label="📹 FPS"
                )
            
            # Generate button
            generate_btn = gr.Button(
                "🎬 Generate Video",
                variant="primary",
                size="lg"
            )
        
        with gr.Column():
            # Output video
            video_output = gr.Video(
                label="🎥 Generated Video",
                autoplay=True,
                show_label=True
            )
            
            # Status message
            status = gr.Textbox(
                label="📊 Status",
                value="Ready to generate!",
                interactive=False
            )
    
    # Example prompts
    gr.Examples(
        examples=[
            ["A serene Japanese garden with cherry blossoms falling gently, koi fish swimming in a crystal clear pond", "blurry, low quality"],
            ["An astronaut floating in space with Earth in the background, cinematic lighting", "distorted, ugly"],
            ["A magical forest with glowing mushrooms and fireflies at night, fantasy style", "bad animation"],
            ["Ocean waves crashing on a rocky beach during golden hour, slow motion", "blurry"],
            ["A cute robot exploring a colorful alien planet, animated style", "low quality"]
        ],
        inputs=[prompt_input, negative_prompt],
        label="💡 Example Prompts"
    )
    
    # Define button click handler
    def on_generate(prompt, neg_prompt, frames, steps, guidance, seed_val, fps_val):
        if not prompt.strip():
            return None, "❌ Please enter a prompt!"
        
        status_msg = f"🎬 Generating video...\nPrompt: {prompt[:50]}..."
        yield None, status_msg
        
        try:
            video_path = generate_video(
                prompt=prompt,
                negative_prompt=neg_prompt,
                num_frames=int(frames),
                num_inference_steps=int(steps),
                guidance_scale=float(guidance),
                seed=int(seed_val),
                fps=int(fps_val)
            )
            return video_path, f"✅ Video generated successfully!\n📁 Saved as: {os.path.basename(video_path)}"
        except Exception as e:
            return None, f"❌ Error: {str(e)}"
    
    # Connect button to function
    generate_btn.click(
        fn=on_generate,
        inputs=[prompt_input, negative_prompt, num_frames, num_steps, guidance_scale, seed, fps],
        outputs=[video_output, status]
    )

print("✅ UI created successfully!")

## 🚀 Step 6: Launch the App

In [ ]:
# Launch Gradio app
print("🚀 Launching Text-to-Video Generator...")
print("📱 Click the link below to open the interface!")
print("\n" + "="*50)
print("🎯 IMPORTANT NOTES:")
print("1. First generation will take 3-5 minutes (model loading)")
print("2. Subsequent generations will be faster (1-2 minutes)")
print("3. Make sure GPU is enabled for best results")
print("4. If you see errors, try restarting the runtime")
print("="*50 + "\n")

demo.launch(
    share=True,  # Creates a public link
    debug=False,
    show_error=True
)

## 📝 Troubleshooting

### Common Issues and Solutions:

#### 1. "CUDA out of memory" error
**Solution:**
- Reduce number of frames (try 8 instead of 16)
- Reduce inference steps (try 15 instead of 25)
- Restart the runtime to clear memory

#### 2. "No module named 'xformers'" error
**Solution:**
- This is not critical, the code will work without it
- The notebook will use default attention instead

#### 3. Model loading takes too long
**Solution:**
- First run always takes 3-5 minutes
- Be patient, the model is being downloaded
- Subsequent runs will be faster

#### 4. Video generation fails
**Solution:**
- Make sure GPU is enabled (Runtime → Change runtime type → GPU)
- Try a simpler prompt
- Reduce the number of frames
- Restart the runtime

#### 5. "Connection reset" or timeout errors
**Solution:**
- This is normal for large model downloads
- Wait a few minutes and try again
- The model will be cached after first download

### Getting Help:
- Check the GPU status in the output above
- Make sure you have at least 8GB VRAM
- Try the example prompts first
- If issues persist, restart the runtime and try again